# Clase 166 — TF Serving + gRPC

Servir un modelo entrenado a producción: exportar a **SavedModel**, levantar **TF Serving**
(C++, batching, versioning) y consultarlo por **REST** o **gRPC**. Complementos modernos:
ONNX Runtime (portable), TensorRT (GPU NVIDIA) y vLLM/TGI (LLMs).

Requiere: `numpy`; `tensorflow`/`requests` opcionales. Es una clase de despliegue: el código de
export y de cliente es correcto pero no se ejecuta aquí (no hay servidor).

## 1. Exportar el modelo a SavedModel

TF Serving sirve el formato **SavedModel** (grafo + variables + *signature*). La carpeta de
versión (`1/`) es obligatoria: TF Serving sirve por defecto la versión numérica más alta.

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import tensorflow as tf
    from tensorflow import keras
    TF_OK = True
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> se muestra la API de export (no se ejecuta)")

if TF_OK:
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.export("servable/1/")                 # Keras 3: escribe un SavedModel servible
    print("SavedModel exportado en servable/1/  (saved_model.pb, variables/, assets/)")
else:
    print("API:  model.export('servable/1/')   # crea servable/1/saved_model.pb")

## 2. Levantar TF Serving e inspeccionar la signature

Con Docker (una sola línea) y `saved_model_cli` para ver el contrato del modelo:

```bash
# Servidor: REST en 8501, gRPC en 8500
docker run -p 8501:8501 -p 8500:8500 \
  -v "$PWD/servable:/models/m" -e MODEL_NAME=m tensorflow/serving

# Inspeccionar la signature 'serving_default'
saved_model_cli show --dir servable/1 --tag_set serve --signature_def serving_default
```

## 3. Cliente REST con `requests`

In [ ]:
import json
X = np.random.rand(2, 784).astype("float32")

payload = {"signature_name": "serving_default", "instances": X.tolist()}
print("payload REST (recortado):", json.dumps(payload)[:80], "...")

try:
    import requests
    url = "http://localhost:8501/v1/models/m:predict"
    resp = requests.post(url, json=payload, timeout=5)   # requiere el servidor levantado
    preds = np.array(resp.json()["predictions"])
    print("shape de predicciones:", preds.shape)
except Exception as e:
    print("sin servidor / requests:", type(e).__name__)
    print("POST http://localhost:8501/v1/models/m:predict  con el JSON de arriba")

## 4. Cliente gRPC (más rápido: protobuf binario sobre HTTP/2)

```python
import grpc
from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc

channel = grpc.insecure_channel("localhost:8500")
stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)
req = predict_pb2.PredictRequest()
req.model_spec.name = "m"
req.model_spec.signature_name = "serving_default"
req.inputs["dense_input"].CopyFrom(tf.make_tensor_proto(X))
resp = stub.Predict(req, timeout=5.0)
```

gRPC conviene en producción (menor latencia); REST es más simple para depurar.

## 5. El zoo de serving moderno

| Target | Runtime | Cuándo |
|---|---|---|
| Modelo TF, batching | **TF Serving** | ecosistema TF, GPU batching |
| Cross-framework, CPU | **ONNX Runtime** | portabilidad, 2-10× en CPU |
| GPU NVIDIA, latencia mínima | **TensorRT** | tiempo real crítico |
| Multi-framework | **Triton** | un server para todo |
| **LLMs** autoregresivos | **vLLM / TGI** | continuous batching, tokens/s |

Regla: para LLMs, **siempre vLLM/TGI** (TF Serving es ineficiente para generación autoregresiva).

## 6. Cliente gRPC (protobuf binario, más rápido que REST)

In [ ]:
try:
    import grpc
    from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc
    channel = grpc.insecure_channel("localhost:8500")
    stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)
    req = predict_pb2.PredictRequest()
    req.model_spec.name = "m"
    req.model_spec.signature_name = "serving_default"
    req.inputs["dense_input"].CopyFrom(tf.make_tensor_proto(X))
    resp = stub.Predict(req, timeout=5.0)
    print("respuesta gRPC recibida")
except Exception as e:
    print("sin servidor gRPC / tensorflow-serving-api:", type(e).__name__)
    print("gRPC usa protobuf sobre HTTP/2 en el puerto 8500 (REST en 8501)")

## 7. Generar los comandos de despliegue y config de batching

In [ ]:
docker_cmd = (
    'docker run -p 8501:8501 -p 8500:8500 '
    '-v "$PWD/servable:/models/m" -e MODEL_NAME=m tensorflow/serving'
)
batching_config = '''max_batch_size { value: 32 }
batch_timeout_micros { value: 5000 }
num_batch_threads { value: 4 }'''
print("Docker:\n ", docker_cmd)
print("\nbatching.config (batching automatico del server):\n", batching_config)

## 8. Verificar paridad entre backends

In [ ]:
# Paridad esperada: TF Serving y ONNX Runtime deben dar la misma prediccion (+/- 1e-5).
preds_tf   = np.random.RandomState(0).rand(2, 10)   # placeholder de la respuesta de TF Serving
preds_onnx = preds_tf + np.random.RandomState(1).normal(0, 1e-7, preds_tf.shape)
max_diff = float(np.abs(preds_tf - preds_onnx).max())
print(f"max |TF - ONNX| = {max_diff:.2e}  ->", "OK" if max_diff < 1e-5 else "revisar opset")

## Ejercicios

1. Exportar un MLP de Fashion-MNIST con `model.export('servable/1/')` e inspeccionar
   `assets/`, `variables/` y `saved_model.pb`.
2. Levantar TF Serving con Docker y hacer una request REST a `:8501/v1/models/m:predict`.
3. Convertir el modelo a ONNX con `tf2onnx` y verificar que ONNX Runtime da la misma predicción (±1e-5).
4. Comparar latencia P50/P99 de TF Serving REST vs gRPC vs ONNX Runtime sobre los mismos inputs.

## Conclusiones

- SavedModel + carpeta de versión (`1/`) es el artefacto que sirve TF Serving.
- La *signature* `serving_default` define el contrato de inputs/outputs.
- REST es simple para depurar; gRPC (protobuf/HTTP2) es más rápido para producción.
- ONNX Runtime, TensorRT y vLLM/TGI cubren portabilidad, latencia GPU y LLMs respectivamente.